# 🏛️ CA Vida — Demonstração Plataforma Snowflake
## Substituição SAS 9.4 → Snowflake | Workflow Solvency II

**Entidade**: CA Vida — Companhia de Seguros de Vida (Grupo Crédito Agrícola)  
**Cenário**: Processo de reporting regulatório Solvency II — já migrado para Snowflake

---
# ACT 1: Platform Foundation
> Compute elástico, ambientes isolados, recuperação instantânea e pipelines como código — tudo nativo, sem ferramentas externas.

## 1.1 — Arquitectura Multi-Ambiente
> Cada equipa tem compute independente. Se o ETL escalar, não afecta a produção. Escalar ou reduzir leva 2 segundos.

In [ ]:
SHOW WAREHOUSES LIKE 'CAVIDA%'

In [ ]:
SHOW SCHEMAS IN DATABASE CAVIDA_DEMO

In [ ]:
-- Escalar compute em 2 segundos
ALTER WAREHOUSE CAVIDA_PROD_WH SET WAREHOUSE_SIZE = 'MEDIUM';
SELECT 'PROD_WH escalado para MEDIUM' AS resultado;
ALTER WAREHOUSE CAVIDA_PROD_WH SET WAREHOUSE_SIZE = 'SMALL'

## 1.2 — Zero-Copy Clone
> Criar um ambiente de teste com TB de dados demora menos de 1 segundo e ocupa zero bytes adicionais de storage.

In [ ]:
CREATE OR REPLACE SCHEMA CAVIDA_DEMO.TEST CLONE CAVIDA_DEMO.PROD
  COMMENT = 'QA environment — zero-copy clone from PROD';

SELECT TABLE_SCHEMA, TABLE_NAME, ROW_COUNT
FROM CAVIDA_DEMO.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA IN ('PROD', 'TEST') AND TABLE_NAME LIKE 'SLV2%'
ORDER BY TABLE_SCHEMA, TABLE_NAME

## 1.3 — Time Travel (90 dias)
> Recuperação instantânea de dados eliminados ou alterados — até 90 dias de histórico automático, sem backups manuais.

In [ ]:
SELECT COUNT(*) AS registos_actuais FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO

In [ ]:
-- Simular eliminação acidental
DELETE FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO WHERE asset_class = 'Cash';
SELECT COUNT(*) AS apos_delete FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO

In [ ]:
-- Time Travel: ver dados ANTES da eliminação
SELECT COUNT(*) AS antes_da_eliminacao
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO AT(OFFSET => -60)

In [ ]:
-- Restaurar instantaneamente
INSERT INTO CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO
SELECT * FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO AT(OFFSET => -60)
WHERE asset_class = 'Cash';
SELECT COUNT(*) AS restaurado FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO

## 1.4 — dbt Project (Pipeline como Código)
> Pipeline de dados versionado, deployed como objecto Snowflake nativo. Estrutura: staging → intermediate → marts. Auditável, reproduzível, testável.

➡️ **Ver no Snowsight**: CAVIDA_DEMO → REGULATORY → dbt Projects → `CAVIDA_SLV2_PROJECT`

In [ ]:
SELECT table_type AS tipo, table_schema AS schema, table_name AS modelo
FROM CAVIDA_DEMO.INFORMATION_SCHEMA.TABLES
WHERE table_schema IN ('DEV_STAGING','DEV_PROD')
ORDER BY table_schema, table_name

## 1.5 — Conectividade & Integração

> Snowflake liga-se a qualquer sistema via drivers nativos, APIs REST, e pipelines de ingestão — sem middleware proprietário.

| Método | Uso | Detalhe |
|--------|-----|---------|
| **JDBC / ODBC** | BI tools (Tableau, Power BI, Qlik), aplicações Java/.NET | Drivers certificados, alta performance, suporte TLS |
| **Python Connector** | Scripts, automação, data science | `snowflake-connector-python`, Snowpark, pandas integration |
| **Spark Connector** | Migração de workloads Spark, integração com Data Lakes | Leitura/escrita nativa entre Spark e Snowflake |
| **Kafka Connector** | Streaming de eventos em tempo real | Snowpipe Streaming, ingestão contínua sub-segundo |
| **Openflow (NiFi)** | Replicação CDC, integração visual com fontes diversas | Drag-and-drop, conectores pré-construídos (Oracle, SAP, APIs) |
| **REST API** | Integração programática, automação, microserviços | SQL API, Snowpark Container Services |
| **PrivateLink** | Conectividade segura sem internet pública | AWS PrivateLink / Azure Private Link — tráfego nunca sai da rede privada |
| **External Stages** | Ingestão de ficheiros S3, Azure Blob, GCS | Snowpipe para auto-ingest com notificação |
| **Partner Connect** | Activação 1-click de ferramentas do ecossistema | dbt Cloud, Fivetran, Informatica, Matillion, Talend |

**Relevância para CA Vida:**
- **Oracle/SQL Server** → JDBC ou Openflow para replicação dos sistemas core
- **Ficheiros Excel/CSV** (carteira investimentos) → Upload via Streamlit ou External Stage
- **Risk Agility / Tagetik** → REST API ou SFTP via External Stage
- **Power BI / Tableau** → ODBC nativo com SSO
- **Plataformas internas CA** → PrivateLink para zero exposição à internet

---
# ACT 2: Governance & Security
> Classificação automática, catálogo nativo, masking dinâmico e segurança por linha — tudo declarativo, sem licenças adicionais.

## 2.1 — Catálogo, Tags & Camada Semântica
> Dados classificados por domínio e sensibilidade. A Semantic View funciona como glossário de negócio — dimensões e métricas documentadas em português.

In [ ]:
SHOW TAGS IN SCHEMA CAVIDA_DEMO.REGULATORY

In [ ]:
SHOW SEMANTIC DIMENSIONS IN CAVIDA_DEMO.SEMANTIC.SLV2_ANALYTICS

In [ ]:
SHOW SEMANTIC METRICS IN CAVIDA_DEMO.SEMANTIC.SLV2_ANALYTICS

## 2.2 — EU Data Residency
> Dados alojados em Frankfurt (AWS EU-Central-1). Encriptação AES-256, edição Business Critical. Conformidade RGPD total.

In [ ]:
SELECT CURRENT_REGION() AS regiao, CURRENT_ACCOUNT_NAME() AS conta

## 2.3 — Dynamic Data Masking
> Mascaramento automático baseado no role do utilizador. Política declarativa aplicada em TODAS as queries — zero código manual.

In [ ]:
-- ACCOUNTADMIN vê dados completos
USE ROLE ACCOUNTADMIN;
SELECT instrument_name, counterparty, rating, market_value
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO LIMIT 5

In [ ]:
-- ANALYST vê dados MASCARADOS
USE ROLE CAVIDA_ANALYST;
USE WAREHOUSE CAVIDA_PROD_WH;
SELECT instrument_name, counterparty, rating, market_value
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO LIMIT 5

In [ ]:
USE ROLE ACCOUNTADMIN

## 2.4 — Row-Level Security
> Segurança invisível: o analista vê apenas dados dos últimos 6 meses sem sequer saber que existe mais. Sem erros, sem filtros visíveis.

In [ ]:
-- ACCOUNTADMIN vê TODOS os períodos
SELECT reference_date, COUNT(*) AS registos
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO
GROUP BY 1 ORDER BY 1

In [ ]:
-- ANALYST só vê últimos 6 meses
USE ROLE CAVIDA_ANALYST;
USE WAREHOUSE CAVIDA_PROD_WH;
SELECT reference_date, COUNT(*) AS registos
FROM CAVIDA_DEMO.RAW.SLV2_INVESTMENT_PORTFOLIO
GROUP BY 1 ORDER BY 1

In [ ]:
USE ROLE ACCOUNTADMIN

---
# ACT 3: Workflow Solvency II — Streamlit App
> Processo de reporting SLV II completo (16 passos) como web app nativa — write-back, upload de ficheiros, validações automáticas. Zero infra-estrutura externa.

➡️ **Abrir no Snowsight**: Streamlit → `CAVIDA_DEMO.REGULATORY.SLV2_WORKFLOW`

In [ ]:
SHOW STREAMLITS IN SCHEMA CAVIDA_DEMO.REGULATORY

---
# ACT 4: FinOps & Cost Intelligence
> Controlo total de custos com pay-per-second, resource monitors com alertas, e custo ZERO quando inactivo. Atribuição exacta por equipa/projecto.

In [ ]:
SHOW RESOURCE MONITORS LIKE 'CAVIDA%'

In [ ]:
SELECT
    warehouse_name AS warehouse,
    ROUND(SUM(credits_used), 2) AS creditos,
    ROUND(SUM(credits_used) * 3.50, 2) AS custo_eur
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE warehouse_name LIKE 'CAVIDA%'
  AND start_time >= DATEADD('day', -30, CURRENT_DATE())
GROUP BY 1 ORDER BY 2 DESC

---
# ACT 5: Snowflake Intelligence — NLQ
> Consultas em linguagem natural sobre os dados SLV II — em português. O modelo gera SQL automaticamente e apresenta resultados com visualizações.

➡️ **Abrir no Snowsight**: Snowflake Intelligence → Agent `CA Vida - Solvency II Intelligence`

**Perguntas sugeridas:**
1. "Qual é o valor total da carteira de investimentos?"
2. "Mostra o balanço económico por tipo de activo"
3. "Qual é o capital disponível por tier?"
4. "Quantos instrumentos temos por classe de activo?"

In [ ]:
SHOW AGENTS IN SCHEMA CAVIDA_DEMO.SEMANTIC

---
# Resumo & Comparativo Final

## 20/20 Requisitos RFI Demonstrados ✅

| Área | Snowflake | Databricks | BigQuery |
|------|-----------|------------|----------|
| **Compute elástico & ambientes** | ✅ Warehouses isolados, auto-suspend, escala em 2s | ⚠️ Clusters always-on, resize lento | ✅ Slots auto, mas sem isolamento por equipa |
| **Zero-Copy Clone** | ✅ < 1 segundo, 0 bytes | ⚠️ Delta Clone (requer compute) | ❌ Não existe |
| **Time Travel** | ✅ 90 dias automáticos | ⚠️ Delta versioning (manual, VACUUM) | ⚠️ 7 dias máximo |
| **Pipeline como código** | ✅ dbt deployed nativo | ⚠️ dbt via Jobs API externo | ⚠️ Dataform (limitado) |
| **Catálogo & glossário** | ✅ Tags + Semantic View nativo | ⚠️ Unity Catalog (recente) | ⚠️ Data Catalog (separado) |
| **EU Data Residency** | ✅ Frankfurt, Business Critical | ✅ Azure/AWS EU disponível | ✅ EU disponível |
| **Dynamic Masking** | ✅ Declarativo, por role | ⚠️ Row/column filters Unity | ⚠️ Policy tags (limitado) |
| **Row-Level Security** | ✅ Invisível, declarativo | ⚠️ Row filters (menos maduro) | ⚠️ Row-level apenas IAM |
| **Web Apps nativas** | ✅ Streamlit in Snowflake | ❌ Não existe | ❌ Não existe |
| **Write-back** | ✅ UPDATE/INSERT directo | ❌ Não existe nativamente | ❌ Não existe nativamente |
| **FinOps** | ✅ Pay-per-second, monitors, zero idle | ❌ DBU caro, clusters ligados | ⚠️ Pay-per-query, sem monitors |
| **NLQ em português** | ✅ Snowflake Intelligence | ❌ Não existe nativamente | ❌ Não existe nativamente |
| **GenAI integrado** | ✅ Cortex (LLMs nativos) | ⚠️ Requer LangChain/externo | ⚠️ Requer Vertex AI |
| **Sector segurador** | ✅ Generali, AXA, Zurich, L&G | ⚠️ Menos referências | ⚠️ Menos referências |

## Próximos Passos

| # | Acção | Prazo |
|---|-------|-------|
| 1 | Workshop hands-on com equipa técnica | 2 semanas |
| 2 | PoC com dados reais SLV II | 4-6 semanas |
| 3 | Migração piloto de 1 pipeline SAS | 6-8 semanas |
| 4 | Plano de migração completo | 8-12 semanas |

---
*Snowflake — The AI Data Cloud*